# Convert Initial Datasets to (Sample ID x Features) Dataframes

This was done for both species and genus-level features.

Initial datasets are as follows:

1) [576 pairs of household and MS patients dataset (Zhou et al, 2023)](https://pmc.ncbi.nlm.nih.gov/articles/PMC10143502/)
2) [129 MS patients + 58 controls (Levi et al, 2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC8080254/)

These dataframes will be saved under `data/preprocessed`.

In [668]:
import pandas as pd
import polars as pl
import re

### Save Species-Level Information

#### Loading Table 1 and Table 5 from first cohort (Zhou)

Table 1 corresponds to the metadata, and Table 5 corresponds to the abundances data.

In [669]:
zhou_meta = pl.read_excel("data/original_datasets/Zhou/Table_S1_Participant_and_samples.xlsx", 
                            sheet_name = "Sample phenotype").to_pandas()

# originally (Features x Samples)
zhou_species = pl.read_excel("data/original_datasets/Zhou/Table_S5_Metagenomics_data.xlsx", 
                            sheet_name = "species").to_pandas()

Features are going to be our columns and participants will be our rows

In [670]:
zhou_species = zhou_species.set_index("Taxa")

# flip axis
zhou_species = zhou_species.T
zhou_species.index.name = "iMSMS_ID" 
zhou_species = zhou_species.reset_index()

merged_samples_species_zhou = zhou_meta.merge(zhou_species, on = "iMSMS_ID", how = "inner")

In [671]:
# save merged df to file - helpful for studying different targets with data later
merged_samples_species_zhou.set_index('iMSMS_ID').to_csv('data/preprocessed/merged_dfs/zhou_species_merged_df.csv')

In [672]:
# comparison of the dataframe shapes
print("Zhou metadata (df1) -", zhou_meta.shape)
print("Zhou species abundances (df2) -", zhou_species.shape)
print("merged", merged_samples_species_zhou.shape)

Zhou metadata (df1) - (1152, 60)
Zhou species abundances (df2) - (1152, 1678)
merged (1152, 1737)


In [673]:
# info on merged dataframe that has metadata and abundance data
print("number of columns:", len(merged_samples_species_zhou.columns)) 
print("number of rows:", len(merged_samples_species_zhou.index)) 
print("names of columns:", merged_samples_species_zhou.columns.tolist())

merged_samples_species = merged_samples_species_zhou.set_index("iMSMS_ID")
print("names of rows", merged_samples_species.index.tolist()) 

number of columns: 1737
number of rows: 1152
names of columns: ['iMSMS_ID', 'household', 'site', 'ethnicity', 'sex', 'age', 'weight', 'height', 'bmi', 'year_of_onset', 'disease', 'disease_course', 'disease_duration', 'treatment_status', 'treatments', 'administration', 'disease_course_control', 'treatment_control', 'treatments_control', 'diet_no_special_needs', 'diet_special_needs', 'spms_year', 'edss', 'MSSS', 'relapse_number', 'csf_results', 'dmt', 'children_number', 'roommates', 'birth_method', 'breastfeeding', 'allergies', 'allergy_specific', 'asthma', 'eating_disorder', 'eczema', 'anxiety', 'manic_depression_bipolar', 'ocd', 'depression', 'postpartum_depression', 'type2diabetes', 'ms_family', 'ocps_y_n', 'which_ocp', 'nsaids', 'nsaids_specifics', 'probiotics', 'rxmeds', 'rxmeds_number', 'otc_meds', 'otc_number', 'smoke', 'recreational_drug', 'weight_change', 'pets', 'education', 'occupation', 'vitamin D (IU)', 'Cohort', 'Megasphaera sp. UPII 135-E', 'Anaerocolumna xylanovorans', 'L

Removal of extra coulmns: 'iMSMS_ID', 'household', 'site', 'ethnicity', 'sex', 'age', 'weight', 'height', 'bmi', 'year_of_onset', 'disease', 'disease_course', 'disease_duration', 'treatment_status', 'treatments', 'administration', 'disease_course_control', 'treatment_control', 'treatments_control', 'diet_no_special_needs', 'diet_special_needs', 'spms_year', 'edss', 'MSSS', 'relapse_number', 'csf_results', 'dmt', 'children_number', 'roommates', 'birth_method', 'breastfeeding', 'allergies', 'allergy_specific', 'asthma', 'eating_disorder', 'eczema', 'anxiety', 'manic_depression_bipolar', 'ocd', 'depression', 'postpartum_depression', 'type2diabetes', 'ms_family', 'ocps_y_n', 'which_ocp', 'nsaids', 'nsaids_specifics', 'probiotics', 'rxmeds', 'rxmeds_number', 'otc_meds', 'otc_number', 'smoke', 'recreational_drug', 'weight_change', 'pets', 'education', 'occupation', 'vitamin D (IU)', 'Cohort'

In [674]:
cohort_pos = merged_samples_species.columns.get_loc("Cohort")
taxa_cols = merged_samples_species.columns[cohort_pos + 1:]
print("number of species features:", len(taxa_cols))

number of species features: 1677


In [675]:
zhou_species_df = merged_samples_species.iloc[:, merged_samples_species.columns.get_loc("Cohort") + 1:]
print(f"Just Species Features Named: {zhou_species_df.columns.tolist()}")

# make sure that all the species feature names are not duplicated
print(f"Number of unique species names: {zhou_species_df.columns.nunique()}")

Just Species Features Named: ['Megasphaera sp. UPII 135-E', 'Anaerocolumna xylanovorans', 'Lactobacillus perolens', 'Bacillus sp. 916', 'Catenibacterium mitsuokai', 'Verrucomicrobia bacterium SCGC AAA164-L15', 'Verrucomicrobia bacterium SCGC AAA164-O14', 'Verrucomicrobia bacterium SCGC AAA168-E21', 'Capnocytophaga gingivalis', 'Solobacterium moorei', 'Streptococcus didelphis', 'Pseudomonas sp. StFLB209', 'Flectobacillus major', 'Prevotella sp. MSX73', 'Brevibacterium senegalense', 'Dielma fastidiosa', 'Actinomyces coleocanis', 'Actinomyces urogenitalis', 'Riemerella columbina', 'Clostridium sp. SY8519', 'Serratia sp. ATCC 39006', 'Lactobacillus frumenti', 'Corynebacterium epidermidicanis', 'Propionibacterium humerusii', 'Ralstonia mannitolilytica', 'Lactobacillus algidus', 'Morococcus cerebrosus', 'Anaerostipes caccae', 'Rhodobacter capsulatus', 'Pseudoflavonifractor capillosus', 'Acinetobacter bereziniae', 'Corynebacterium uterequi', 'Candidatus Burkholderia crenata', 'Rhodopseudomona

Final Species (Zhou) DF for Project:

In [676]:
zhou_species_df.head()

,Megasphaera sp. UPII 135-E,Anaerocolumna xylanovorans,Lactobacillus perolens,Bacillus sp. 916,Catenibacterium mitsuokai,Verrucomicrobia bacterium SCGC AAA164-L15,Verrucomicrobia bacterium SCGC AAA164-O14,Verrucomicrobia bacterium SCGC AAA168-E21,Capnocytophaga gingivalis,Solobacterium moorei,...,Oceanospirillum linum,Selenomonas ruminantium,Clostridium sp. ASF356,Eubacterium plexicaudatum,Lactobacillus mucosae,Roseimaritima ulvae,Saccharicrinis fermentans,Bifidobacterium reuteri,Lactobacillus kimchiensis,Tetrasphaera japonica
iMSMS_ID,,,,,,,,,,,,,,,,,,,,,
71401-0001,0.000013,7.563795e-06,0.000000e+00,0.0,0.000007,0.0,0.0,0.0,0.00000,0.000237,...,0.000000e+00,0.000002,0.000003,0.000074,0.000023,0.000000,1.163661e-06,0.000012,0.0,0.0
71401-0009,0.000038,4.471102e-07,0.000000e+00,0.0,0.000005,0.0,0.0,0.0,0.00000,0.000415,...,4.471102e-07,0.000000,0.000003,0.000112,0.000004,0.000000,4.471102e-07,0.000044,0.0,0.0
71401-0017,0.000028,1.509117e-06,5.030389e-07,0.0,0.000016,0.0,0.0,0.0,0.00000,0.000527,...,0.000000e+00,0.000000,0.000002,0.000065,0.000415,0.000000,0.000000e+00,0.000026,0.0,0.0
71401-0022,0.000005,8.353277e-07,0.000000e+00,0.0,0.000005,0.0,0.0,0.0,0.00002,0.000038,...,1.252992e-05,0.000000,0.000000,0.000079,0.000000,0.000002,0.000000e+00,0.000000,0.0,0.0
71401-0027,0.000011,0.000000e+00,0.000000e+00,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000116,...,2.152702e-06,0.000001,0.000000,0.000041,0.000019,0.000000,0.000000e+00,0.000029,0.0,0.0


#### Loading mmc3 and mmc4 sheets from second cohort (Levi)

`mmc3.xlsx` contains the sample metadata, while `mmc4.xlsx` contains the abundance data.

In [677]:
levi_meta = pl.read_excel("data/original_datasets/Levi/mmc3.xlsx", 
                               sheet_name = "Microbiome particpants").to_pandas()

# the genus + species level data are both under the same sheet
levi_species = pl.read_excel("data/original_datasets/Levi/mmc4.xlsx", 
                               sheet_name = "Species").to_pandas()

Merging on FD -- mmc3 sheet has FD combined with Registrant number for its Sample ID

In [678]:
id_col = levi_species.columns[0]
levi_species[id_col] = levi_species[id_col].astype(str).str.split("_").str[-1].str.strip()
levi_species = levi_species.rename(columns = {id_col:"FD"})

In [679]:
merged_samples_species_levi = levi_meta.merge(levi_species, on = "FD", how = "inner")

In [680]:
print("Levi meta (df1) -", levi_meta.shape)
print("Levi abundances (df2) -", levi_species.shape)
print("merged", merged_samples_species_levi.shape)

Levi meta (df1) - (187, 21)
Levi abundances (df2) - (187, 687)
merged (187, 707)


In [681]:
print("number of columns:", len(merged_samples_species_levi.columns)) 
print("number of rows:", len(merged_samples_species_levi.index)) 
print("names of columns:", merged_samples_species_levi.columns.tolist())

number of columns: 707
number of rows: 187
names of columns: ['Registrant_number', 'sample_date', 'FD', 'Have serum gene sample', 'Recorded nutritional for at least 3 days', 'type', 'age', 'gender', 'BMI', 'diagnosis_date', 'treatment_group', 'treatment_start', 'treatment_end', 'treatment_duration', 'Registrant connected', 'Connection type', 'MS_type short', 'One_Relapse_or_more_during_following_year', 'worse_EDSS_after_one_year', 'worse_EDSS_after_one_year.1', 'Previous treatments', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_marmotae|fSGB__2483|gSGB__6985|sSGB__10064', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli|fSGB__2483|gSGB__6985|sSGB__10068', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Klebsiella|s__Klebsiella_pneumoniae|fSGB__2483|gSGB__6998|sSGB__10115', 'k

In [682]:
merged_samples_species_levi = merged_samples_species_levi.set_index("FD") 
print("names of rows", merged_samples_species_levi.index.tolist())

names of rows ['FD2917', 'FD2387', 'FD2901', 'FD2878', 'FD2298', 'FD2939', 'FD2912', 'FD2287', 'FD2369', 'FD2341', 'FD2880', 'FD2345', 'FD2894', 'FD2654', 'FD1888', 'FD2379', 'FD2358', 'FD2288', 'FD2652', 'FD2663', 'FD2657', 'FD2615', 'FD2627', 'FD2881', 'FD2823', 'FD2662', 'FD2375', 'FD2630', 'FD2920', 'FD1885', 'FD2324', 'FD2874', 'FD2389', 'FD2829', 'FD2363', 'FD1878', 'FD2625', 'FD2317', 'FD2330', 'FD2841', 'FD2902', 'FD2385', 'FD2383', 'FD2320', 'FD1876', 'FD1905', 'FD1867', 'FD2326', 'FD2635', 'FD2824', 'FD2373', 'FD2354', 'FD2626', 'FD2634', 'FD2825', 'FD2953', 'FD2956', 'FD2325', 'FD1882', 'FD2938', 'FD2331', 'FD2639', 'FD2368', 'FD2651', 'FD2284', 'FD2292', 'FD2297', 'FD2303', 'FD2306', 'FD2312', 'FD2329', 'FD2333', 'FD2335', 'FD2340', 'FD2339', 'FD2344', 'FD2342', 'FD2348', 'FD1860', 'FD1864', 'FD1866', 'FD1868', 'FD1871', 'FD1877', 'FD1890', 'FD1891', 'FD1894', 'FD1906', 'FD1908', 'FD1896', 'FD1899', 'FD1909', 'FD1913', 'FD1898', 'FD1904', 'FD1907', 'FD1901', 'FD1911', 'FD23

Removing extra columns: 

In [683]:
# get the index of where the metadata ends
pt_pos = merged_samples_species_levi.columns.get_loc("Previous treatments")

# take only the taxa abundance data
taxa_cols_02 = merged_samples_species_levi.columns[pt_pos + 1:]
print("number of species features:", len(taxa_cols_02))

number of species features: 686


Almost Final Species (Levi) DF for Project:

In [684]:
levi_species_df = merged_samples_species_levi.iloc[:, merged_samples_species_levi.columns.get_loc("Previous treatments") + 1:]
levi_species_df.head()

,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_marmotae|fSGB__2483|gSGB__6985|sSGB__10064,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli|fSGB__2483|gSGB__6985|sSGB__10068,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Klebsiella|s__Klebsiella_pneumoniae|fSGB__2483|gSGB__6998|sSGB__10115,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Enterobacter|s__Enterobacter_cloacae|fSGB__2483|gSGB__6999|sSGB__10130,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_568|fSGB__343|gSGB__781|sSGB__1024,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_349|fSGB__2833|gSGB__9059|sSGB__13976,k__unknown|p__unknown|c__unknown|o__unknown|f__unknown|g__unknown|s__unknown|fSGB__2834|gSGB__9060|sSGB__13977,k__unknown|p__unknown|c__unknown|o__unknown|f__unknown|g__unknown|s__unknown|fSGB__2834|gSGB__9061|sSGB__13979,k__unknown|p__unknown|c__unknown|o__unknown|f__unknown|g__unknown|s__unknown|fSGB__2834|gSGB__9062|sSGB__13981,k__unknown|p__unknown|c__unknown|o__unknown|f__unknown|g__unknown|s__unknown|fSGB__2834|gSGB__9063|sSGB__13982,...,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__unknown|s__unknown|fSGB__2404|gSGB__6610|sSGB__9344,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__Azospirillum|s__Azospirillum_sp_CAG_239|fSGB__2404|gSGB__6612|sSGB__9346,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__Azospirillum|s__Azospirillum_sp_CAG_260|fSGB__2404|gSGB__6613|sSGB__9347,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Oxalobacteraceae|g__unknown|s__unknown|fSGB__2429|gSGB__6649|sSGB__9391,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Coriobacteriaceae|g__Coriobacteriaceae_unclassified|s__Coriobacteriaceae_bacterium_BV3Ac1|fSGB__324|gSGB__745|sSGB__960,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__unknown|fSGB__2477|gSGB__6812|sSGB__9699,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__Haemophilus_sp_HMSC061E01|fSGB__2477|gSGB__6812|sSGB__9701,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__Haemophilus_parainfluenzae|fSGB__2477|gSGB__6812|sSGB__9712,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Atopobiaceae|g__Atopobium|s__Atopobium_minutum|fSGB__328|gSGB__751|sSGB__987,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Atopobiaceae|g__Atopobium|s__Atopobium_vaginae|fSGB__329|gSGB__755|sSGB__991
FD,,,,,,,,,,,,,,,,,,,,,
FD2917,0.000100,0.000270,0.0001,0.0001,0.000100,0.024233,0.0001,0.0001,0.000100,0.000287,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2387,0.000100,0.000244,0.0001,0.0001,0.000162,0.000100,0.0001,0.0001,0.000403,0.000100,...,0.000100,0.003511,0.001487,0.000411,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2901,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000185,...,0.005789,0.002430,0.003647,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2878,0.001432,0.109726,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.002455
FD2298,0.000212,0.004942,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,...,0.000100,0.000100,0.000100,0.000100,0.000659,0.0001,0.0001,0.0001,0.0001,0.000100


In [685]:
print("Just Species Features Named:", levi_species_df.columns.tolist())

Just Species Features Named: ['k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_marmotae|fSGB__2483|gSGB__6985|sSGB__10064', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli|fSGB__2483|gSGB__6985|sSGB__10068', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Klebsiella|s__Klebsiella_pneumoniae|fSGB__2483|gSGB__6998|sSGB__10115', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Enterobacter|s__Enterobacter_cloacae|fSGB__2483|gSGB__6999|sSGB__10130', 'k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_568|fSGB__343|gSGB__781|sSGB__1024', 'k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_349|fSGB__2833|gSGB__9059|

In [686]:
def standardize_species(names):

    standardized_names = [] 

    for name in names:

        match = re.search(r's__([^|]+)', name) #help here from AI with regex match -- better standardizing will be needed -- some species names have trailing letter symbols

        if match:
            species = match.group(1)
        else:
            species = name #no s__ pattern 

        species = re.sub(r'^([A-Za-z]+)_[A-Z]+_', r'\1_', species)

        species = species.replace('_', ' ').strip() 

        standardized_names.append(species)

    return standardized_names

In [687]:
# standardize the species names
levi_species_df.columns = standardize_species(levi_species_df.columns.tolist())
levi_species_df.head()

,Escherichia marmotae,Escherichia coli,Klebsiella pneumoniae,Enterobacter cloacae,Clostridium sp CAG 568,Clostridium sp CAG 349,unknown,unknown,unknown,unknown,...,unknown,Azospirillum sp CAG 239,Azospirillum sp CAG 260,unknown,Coriobacteriaceae bacterium BV3Ac1,unknown,Haemophilus sp HMSC061E01,Haemophilus parainfluenzae,Atopobium minutum,Atopobium vaginae
FD,,,,,,,,,,,,,,,,,,,,,
FD2917,0.000100,0.000270,0.0001,0.0001,0.000100,0.024233,0.0001,0.0001,0.000100,0.000287,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2387,0.000100,0.000244,0.0001,0.0001,0.000162,0.000100,0.0001,0.0001,0.000403,0.000100,...,0.000100,0.003511,0.001487,0.000411,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2901,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000185,...,0.005789,0.002430,0.003647,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
FD2878,0.001432,0.109726,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.002455
FD2298,0.000212,0.004942,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.000100,0.000100,...,0.000100,0.000100,0.000100,0.000100,0.000659,0.0001,0.0001,0.0001,0.0001,0.000100


In [688]:
# drop the unknown species columns
levi_species_known_df = levi_species_df.drop(columns=['unknown'])
print(f"Number of features left that were not unknown: {len(levi_species_known_df.columns)}")

Number of features left that were not unknown: 369


In [689]:
# need to drop any duplicates species columns as well
levi_species_unique_df = levi_species_known_df.loc[:, ~levi_species_known_df.columns.duplicated()]

print(f"Final number of leftover, unique features: {len(levi_species_unique_df.columns)}")

Final number of leftover, unique features: 318


### Save Genus-Level Information

In [690]:
# make copy of metadata and read in genus data sheet
zhou_meta_copy = zhou_meta.copy()
zhou_genus = pl.read_excel("data/original_datasets/Zhou/Table_S5_Metagenomics_data.xlsx", 
                          sheet_name = "genus").to_pandas()

In [691]:
# readjust the index name
zhou_genus = zhou_genus.set_index("Taxa")

zhou_genus = zhou_genus.T
zhou_genus.index.name = "iMSMS_ID"
zhou_genus = zhou_genus.reset_index()

In [692]:
# standardize genus column names first
zhou_standardized = [col.lower().strip() for col in zhou_genus.columns[1:]]
zhou_standardized.insert(0, 'iMSMS_ID')

zhou_genus.columns = zhou_standardized
zhou_genus.head()

,iMSMS_ID,cellvibrio,coprobacillus,flectobacillus,capnocytophaga,pseudoflavonifractor,beggiatoa,collinsella,candidatus marithrix,rhodobacter,...,nitrosomonas,aquabacterium,roseateles,flavonifractor,bdellovibrio,herbaspirillum,oceanospirillum,selenomonas,christensenella,tetrasphaera
0,71401-0001,0.0,0.000361,0.0,7.320488e-06,0.002184,0.000000,0.002537,0.0,0.0,...,0.0,6.100407e-07,0.0,0.000150,0.0,0.000000,0.0,0.000048,0.000006,0.0
1,71401-0002,0.0,0.001862,0.0,8.442089e-07,0.000578,0.000000,0.005956,0.0,0.0,...,0.0,0.000000e+00,0.0,0.007752,0.0,0.000000,0.0,0.000015,0.000000,0.0
2,71401-0003,0.0,0.000793,0.0,1.665922e-05,0.000795,0.000004,0.000623,0.0,0.0,...,0.0,0.000000e+00,0.0,0.000092,0.0,0.000001,0.0,0.000011,0.000000,0.0
3,71401-0004,0.0,0.000522,0.0,3.744612e-06,0.001237,0.000001,0.000180,0.0,0.0,...,0.0,0.000000e+00,0.0,0.000106,0.0,0.000000,0.0,0.000038,0.000004,0.0
4,71401-0005,0.0,0.000558,0.0,5.689114e-06,0.000345,0.000002,0.000162,0.0,0.0,...,0.0,0.000000e+00,0.0,0.009716,0.0,0.000000,0.0,0.000010,0.000006,0.0


In [693]:
# merge the metadata and genus abundances together
merged_samples_genus_zhou = zhou_meta_copy.merge(zhou_genus, on = "iMSMS_ID", how = "inner")

In [694]:
# compare dataframe shapes
print("samples shape:", zhou_meta_copy.shape)
print("genus shape:", zhou_genus.shape)
print("merged shape:", merged_samples_genus_zhou.shape)

samples shape: (1152, 60)
genus shape: (1152, 547)
merged shape: (1152, 606)


In [695]:
# look at how many columns and rows the merged DF has
print("number of columns:", len(merged_samples_genus_zhou.columns)) 
print("number of rows:", len(merged_samples_genus_zhou.index)) 
print("names of columns:", merged_samples_genus_zhou.columns.tolist()) 

merged_samples_genus_zhou = merged_samples_genus_zhou.set_index("iMSMS_ID")
print("names of rows", merged_samples_genus_zhou.index.tolist())

number of columns: 606
number of rows: 1152
names of columns: ['iMSMS_ID', 'household', 'site', 'ethnicity', 'sex', 'age', 'weight', 'height', 'bmi', 'year_of_onset', 'disease', 'disease_course', 'disease_duration', 'treatment_status', 'treatments', 'administration', 'disease_course_control', 'treatment_control', 'treatments_control', 'diet_no_special_needs', 'diet_special_needs', 'spms_year', 'edss', 'MSSS', 'relapse_number', 'csf_results', 'dmt', 'children_number', 'roommates', 'birth_method', 'breastfeeding', 'allergies', 'allergy_specific', 'asthma', 'eating_disorder', 'eczema', 'anxiety', 'manic_depression_bipolar', 'ocd', 'depression', 'postpartum_depression', 'type2diabetes', 'ms_family', 'ocps_y_n', 'which_ocp', 'nsaids', 'nsaids_specifics', 'probiotics', 'rxmeds', 'rxmeds_number', 'otc_meds', 'otc_number', 'smoke', 'recreational_drug', 'weight_change', 'pets', 'education', 'occupation', 'vitamin D (IU)', 'Cohort', 'cellvibrio', 'coprobacillus', 'flectobacillus', 'capnocytophag

In [696]:
# save merged df to file - may be helpful for studying different targets with data later
merged_samples_genus_zhou.to_csv('data/preprocessed/merged_dfs/zhou_genus_merged_df.csv')

merged_samples_genus_zhou.head()

,household,site,ethnicity,sex,age,weight,height,bmi,year_of_onset,disease,...,nitrosomonas,aquabacterium,roseateles,flavonifractor,bdellovibrio,herbaspirillum,oceanospirillum,selenomonas,christensenella,tetrasphaera
iMSMS_ID,,,,,,,,,,,,,,,,,,,,,
71401-0001,7140001,San Francisco,Caucasian/Hispanic,M,54.0,90.70,187.96,25.672988,2011.0,MS,...,0.0,6.100407e-07,0.0,0.000150,0.0,0.000000e+00,0.000000e+00,0.000048,6.100407e-06,0.0
71401-0009,7140009,San Francisco,Caucasian/Hispanic,F,35.0,83.00,177.80,26.255155,2012.0,MS,...,0.0,0.000000e+00,0.0,0.000735,0.0,4.637378e-07,4.637378e-07,0.000004,1.391213e-06,0.0
71401-0017,7140017,San Francisco,Caucasian/Hispanic,F,38.0,61.03,167.03,21.875320,1998.0,MS,...,0.0,0.000000e+00,0.0,0.011209,0.0,0.000000e+00,0.000000e+00,0.000015,5.273213e-07,0.0
71401-0022,7140022,San Francisco,Caucasian/Hispanic,M,69.0,74.85,185.83,21.675052,1998.0,MS,...,0.0,1.254317e-05,0.0,0.000447,0.0,0.000000e+00,1.343911e-05,0.000004,0.000000e+00,0.0
71401-0027,7140027,San Francisco,Caucasian/Hispanic,F,45.0,53.52,157.18,21.663146,1997.0,MS,...,0.0,0.000000e+00,0.0,0.000689,0.0,0.000000e+00,2.213332e-06,0.000015,1.106666e-06,0.0


In [697]:
cohort_pos = merged_samples_genus_zhou.columns.get_loc("Cohort")
taxa_cols = merged_samples_genus_zhou.columns[cohort_pos + 1:]
print("number of genus features:", len(taxa_cols))

number of genus features: 546


Again, removing features/columns not needed:

In [698]:
# get only the abundance data from the merged DF, without metadata columns
zhou_genus_df = merged_samples_genus_zhou.iloc[:, merged_samples_genus_zhou.columns.get_loc("Cohort") + 1:]
print(f"Just Genus Features Named: {zhou_genus_df.columns.tolist()}")

# double check that all features are actually unique
print(f"Number of unique genus features: {zhou_genus_df.columns.nunique()}")

Just Genus Features Named: ['cellvibrio', 'coprobacillus', 'flectobacillus', 'capnocytophaga', 'pseudoflavonifractor', 'beggiatoa', 'collinsella', 'candidatus marithrix', 'rhodobacter', 'cupriavidus', 'trueperella', 'rhodopseudomonas', 'methanomassiliicoccus', 'rhodospirillum', 'lonsdalea', 'desulfotalea', 'anaerovorax', 'leifsonia', 'microcystis', 'gemmata', 'pseudoramibacter', 'candidatus magnetomorum', 'tepidimonas', 'alkaliphilus', 'proteiniclasticum', 'parasporobacterium', 'macellibacteroides', 'helcobacillus', 'murdochiella', 'lachnoanaerobaculum', 'aphanizomenon', 'granulicatella', 'defluviitalea', 'bulleidia', 'gilliamella', 'nitrolancea', 'dyadobacter', 'stomatobaculum', 'pectobacterium', 'solobacterium', 'piscirickettsia', 'brackiella', 'leuconostoc', 'pediococcus', 'peptostreptococcus', 'ruminococcus', 'micrococcus', 'anaerosphaera', 'staphylococcus', 'eggerthia', 'kandleria', 'alloprevotella', 'acidovorax', 'phaseolibacter', 'deinococcus', 'carboxydothermus', 'streptococcus

Final Genus DF for Project:

In [699]:
zhou_genus_df.head()

,cellvibrio,coprobacillus,flectobacillus,capnocytophaga,pseudoflavonifractor,beggiatoa,collinsella,candidatus marithrix,rhodobacter,cupriavidus,...,nitrosomonas,aquabacterium,roseateles,flavonifractor,bdellovibrio,herbaspirillum,oceanospirillum,selenomonas,christensenella,tetrasphaera
iMSMS_ID,,,,,,,,,,,,,,,,,,,,,
71401-0001,0.0,0.000361,0.0,0.000007,0.002184,0.000000,0.002537,0.000000e+00,0.0,0.0,...,0.0,6.100407e-07,0.0,0.000150,0.0,0.000000e+00,0.000000e+00,0.000048,6.100407e-06,0.0
71401-0009,0.0,0.000084,0.0,0.000006,0.000486,0.000000,0.001526,0.000000e+00,0.0,0.0,...,0.0,0.000000e+00,0.0,0.000735,0.0,4.637378e-07,4.637378e-07,0.000004,1.391213e-06,0.0
71401-0017,0.0,0.000245,0.0,0.000010,0.000388,0.000000,0.000037,0.000000e+00,0.0,0.0,...,0.0,0.000000e+00,0.0,0.011209,0.0,0.000000e+00,0.000000e+00,0.000015,5.273213e-07,0.0
71401-0022,0.0,0.000303,0.0,0.000030,0.000213,0.000000,0.000414,8.959409e-07,0.0,0.0,...,0.0,1.254317e-05,0.0,0.000447,0.0,0.000000e+00,1.343911e-05,0.000004,0.000000e+00,0.0
71401-0027,0.0,0.000289,0.0,0.000019,0.000920,0.000002,0.008999,0.000000e+00,0.0,0.0,...,0.0,0.000000e+00,0.0,0.000689,0.0,0.000000e+00,2.213332e-06,0.000015,1.106666e-06,0.0


#### Also save genus-level data for Levi dataset

In [700]:
levi_meta_copy = levi_meta.copy()
levi_genus = levi_species.copy()

id_col = levi_genus.columns[0]
levi_genus[id_col] = levi_genus[id_col].astype(str).str.split("_").str[-1].str.strip() #Fixing Column name for join
levi_genus = levi_genus.rename(columns = {id_col:"FD"})

merged_samples_genus_levi = levi_meta_copy.merge(levi_genus, on = "FD", how = "inner")
merged_samples_genus_levi.head()

,Registrant_number,sample_date,FD,Have serum gene sample,Recorded nutritional for at least 3 days,type,age,gender,BMI,diagnosis_date,...,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__unknown|s__unknown|fSGB__2404|gSGB__6610|sSGB__9344,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__Azospirillum|s__Azospirillum_sp_CAG_239|fSGB__2404|gSGB__6612|sSGB__9346,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Rhodospirillales|f__Rhodospirillaceae|g__Azospirillum|s__Azospirillum_sp_CAG_260|fSGB__2404|gSGB__6613|sSGB__9347,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Oxalobacteraceae|g__unknown|s__unknown|fSGB__2429|gSGB__6649|sSGB__9391,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Coriobacteriaceae|g__Coriobacteriaceae_unclassified|s__Coriobacteriaceae_bacterium_BV3Ac1|fSGB__324|gSGB__745|sSGB__960,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__unknown|fSGB__2477|gSGB__6812|sSGB__9699,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__Haemophilus_sp_HMSC061E01|fSGB__2477|gSGB__6812|sSGB__9701,k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Pasteurellales|f__Pasteurellaceae|g__Haemophilus|s__Haemophilus_parainfluenzae|fSGB__2477|gSGB__6812|sSGB__9712,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Atopobiaceae|g__Atopobium|s__Atopobium_minutum|fSGB__328|gSGB__751|sSGB__987,k__Bacteria|p__Actinobacteria|c__Coriobacteriia|o__Coriobacteriales|f__Atopobiaceae|g__Atopobium|s__Atopobium_vaginae|fSGB__329|gSGB__755|sSGB__991
0,32313,2018-08-08,FD2917,No,Yes,MS,50.40,female,29,2004-03-16,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
1,911500,2017-06-07,FD2387,No,No,MS,53.41,female,21,2003-12-18,...,0.000100,0.003511,0.001487,0.000411,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
2,976854,2018-07-16,FD2901,No,No,MS,57.15,male,24,1987-01-01,...,0.005789,0.002430,0.003647,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.000100
3,235772,2018-05-15,FD2878,No,No,MS,42.86,female,20,1998-11-01,...,0.000100,0.000100,0.000100,0.000100,0.000100,0.0001,0.0001,0.0001,0.0001,0.002455
4,633807,2016-02-18,FD2298,Yes,Yes,MS,44.90,female,24,2011-02-01,...,0.000100,0.000100,0.000100,0.000100,0.000659,0.0001,0.0001,0.0001,0.0001,0.000100


In [701]:
# compare the dataframe shapes
print("Levi metadata df copy (df1) -", levi_meta_copy.shape)
print("Levi genus df copy (df2) -", levi_genus.shape)
print("merged", merged_samples_genus_levi.shape)

Levi metadata df copy (df1) - (187, 21)
Levi genus df copy (df2) - (187, 687)
merged (187, 707)


In [702]:
# study the dimensions of the merged DF
print("number of columns:", len(merged_samples_genus_levi.columns)) 
print("number of row:", len(merged_samples_genus_levi.index)) 
print("names of columns:", merged_samples_genus_levi.columns.tolist())

merged_samples_genus_levi = merged_samples_genus_levi.set_index("FD") 
print("names of rows", merged_samples_genus_levi.index.tolist())

number of columns: 707
number of row: 187
names of columns: ['Registrant_number', 'sample_date', 'FD', 'Have serum gene sample', 'Recorded nutritional for at least 3 days', 'type', 'age', 'gender', 'BMI', 'diagnosis_date', 'treatment_group', 'treatment_start', 'treatment_end', 'treatment_duration', 'Registrant connected', 'Connection type', 'MS_type short', 'One_Relapse_or_more_during_following_year', 'worse_EDSS_after_one_year', 'worse_EDSS_after_one_year.1', 'Previous treatments', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_marmotae|fSGB__2483|gSGB__6985|sSGB__10064', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli|fSGB__2483|gSGB__6985|sSGB__10068', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Klebsiella|s__Klebsiella_pneumoniae|fSGB__2483|gSGB__6998|sSGB__10115', 'k_

In [703]:
# see the number of genus feature columns
pt_pos = merged_samples_genus_levi.columns.get_loc("Previous treatments")
taxa_cols_02 = merged_samples_genus_levi.columns[pt_pos + 1:]
print("number of genus features (not necessarily all unique):", len(taxa_cols_02))

number of genus features (not necessarily all unique): 686


In [704]:
# get only genus feature names
merged_samples_genus_levi = merged_samples_genus_levi.iloc[:, merged_samples_genus_levi.columns.get_loc("Previous treatments") + 1:]
print("Just Genus Features Named:", merged_samples_genus_levi.columns.tolist())

Just Genus Features Named: ['k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_marmotae|fSGB__2483|gSGB__6985|sSGB__10064', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli|fSGB__2483|gSGB__6985|sSGB__10068', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Klebsiella|s__Klebsiella_pneumoniae|fSGB__2483|gSGB__6998|sSGB__10115', 'k__Bacteria|p__Proteobacteria|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Enterobacter|s__Enterobacter_cloacae|fSGB__2483|gSGB__6999|sSGB__10130', 'k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_568|fSGB__343|gSGB__781|sSGB__1024', 'k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_sp_CAG_349|fSGB__2833|gSGB__9059|sS

In [705]:
def standardize_genus(name):
    # get genus name
    g__split = name.split("g__")[1]
    genus = g__split.split('|')[0]
        
    # get rid of any brackets and make all spaces underscore
    genus = genus.replace('[', '').replace(']', '')
    genus = genus.replace(' ', '_').lower().strip()
    
    return genus

In [706]:
levi_genus_df = levi_genus.set_index('FD')
levi_genus_df.columns = pd.Series(levi_genus_df.columns).apply(standardize_genus)
levi_genus_df.head()

,escherichia,escherichia,klebsiella,enterobacter,clostridium,clostridium,unknown,unknown,unknown,unknown,...,unknown,azospirillum,azospirillum,unknown,coriobacteriaceae_unclassified,haemophilus,haemophilus,haemophilus,atopobium,atopobium
FD,,,,,,,,,,,,,,,,,,,,,
FD2664,0.000100,0.052562,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.0001,0.000203,...,0.0001,0.014237,0.000100,0.0001,0.000100,0.0001,0.0001,0.0001,0.000100,0.00010
FD2364,0.000863,0.025337,0.0001,0.0001,0.030225,0.001979,0.0001,0.0001,0.0001,0.000100,...,0.0001,0.000100,0.000100,0.0001,0.000100,0.0001,0.0001,0.0001,0.000128,0.00010
FD2881,0.001047,0.019842,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.0001,0.001240,...,0.0001,0.000100,0.027688,0.0001,0.000100,0.0001,0.0001,0.0001,0.000100,0.00541
FD2627,0.000100,0.000143,0.0001,0.0001,0.000100,0.000100,0.0001,0.0001,0.0001,0.000605,...,0.0001,0.002692,0.000100,0.0001,0.000967,0.0001,0.0001,0.0001,0.000100,0.00010
FD2917,0.000100,0.000270,0.0001,0.0001,0.000100,0.024233,0.0001,0.0001,0.0001,0.000287,...,0.0001,0.000100,0.000100,0.0001,0.000100,0.0001,0.0001,0.0001,0.000100,0.00010


In [707]:
# drop the unknown genus columns
levi_genus_known_df = levi_genus_df.drop(columns=['unknown'])
print(f"Number of genus left that were not unknown: {len(levi_genus_known_df.columns)}")

# need to drop any duplicates genus columns as well
levi_genus_unique_df = levi_genus_known_df.loc[:, ~levi_genus_known_df.columns.duplicated()]
print(f"Final number of leftover, unique genus: {len(levi_genus_unique_df.columns)}")

Number of genus left that were not unknown: 489
Final number of leftover, unique genus: 103


### Set all the Pseudocount-Type Abundances (0.0001) to 0 in Levi Dataset

In [708]:
levi_species_unique_df = levi_species_unique_df.mask(levi_species_unique_df <= 0.0001, 0)

levi_genus_unique_df = levi_genus_unique_df.mask(levi_genus_unique_df <= 0.0001, 0)

levi_genus_unique_df.head()

,escherichia,klebsiella,enterobacter,clostridium,subdoligranulum,ruminococcus,firmicutes_unclassified,eggerthella,cryptobacterium,olsenella,...,coraliomargarita,akkermansia,proteobacteria_unclassified,burkholderiales_unclassified,sutterella,acetobacter,azospirillum,coriobacteriaceae_unclassified,haemophilus,atopobium
FD,,,,,,,,,,,,,,,,,,,,,
FD2664,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000323,0.0,...,0.000000,0.000518,0.000000,0.003227,0.000000,0.00000,0.014237,0.000000,0.0,0.000000
FD2364,0.000863,0.0,0.0,0.030225,0.000000,0.000265,0.0,0.000209,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.000128
FD2881,0.001047,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000478,0.0,...,0.000000,0.000000,0.000000,0.001863,0.000000,0.00000,0.000000,0.000000,0.0,0.000000
FD2627,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000572,0.0,...,0.000000,0.000000,0.000157,0.006400,0.002148,0.00000,0.002692,0.000967,0.0,0.000000
FD2917,0.000000,0.0,0.0,0.000000,0.001109,0.000000,0.0,0.000470,0.000000,0.0,...,0.000149,0.000000,0.000000,0.000123,0.000000,0.02859,0.000000,0.000000,0.0,0.000000


### Write All Data to CSV:

In [709]:
# all abundance data
zhou_species_df.to_csv("data/preprocessed/abundance_data/Cohort1_Zhou_Species.csv")
levi_species_unique_df.to_csv("data/preprocessed/abundance_data/Cohort2_Levi_Species.csv")

zhou_genus_df.to_csv("data/preprocessed/abundance_data/Cohort1_Zhou_Genus.csv")
levi_genus_unique_df.to_csv("data/preprocessed/abundance_data/Cohort2_Levi_Genus.csv")

In [710]:
# merged Levi datasets that include all metadata along with abundances

# species-level
merged_species = levi_meta.merge(levi_species_unique_df, on = "FD", how = "inner")
merged_species.to_csv('data/preprocessed/merged_dfs/levi_species_merged_df.csv', index=False)
merged_species.head()

# genus-level
merged_genus = levi_meta.merge(levi_genus_unique_df, on = "FD", how = "inner")
merged_genus.to_csv('data/preprocessed/merged_dfs/levi_genus_merged_df.csv', index=False)
merged_genus.head()

,Registrant_number,sample_date,FD,Have serum gene sample,Recorded nutritional for at least 3 days,type,age,gender,BMI,diagnosis_date,...,coraliomargarita,akkermansia,proteobacteria_unclassified,burkholderiales_unclassified,sutterella,acetobacter,azospirillum,coriobacteriaceae_unclassified,haemophilus,atopobium
0,32313,2018-08-08,FD2917,No,Yes,MS,50.40,female,29,2004-03-16,...,0.000149,0.000000,0.0,0.000123,0.0,0.028590,0.000000,0.000000,0.0,0.0
1,911500,2017-06-07,FD2387,No,No,MS,53.41,female,21,2003-12-18,...,0.001953,0.000168,0.0,0.000217,0.0,0.006065,0.003511,0.000000,0.0,0.0
2,976854,2018-07-16,FD2901,No,No,MS,57.15,male,24,1987-01-01,...,0.000115,0.000000,0.0,0.000117,0.0,0.003849,0.002430,0.000000,0.0,0.0
3,235772,2018-05-15,FD2878,No,No,MS,42.86,female,20,1998-11-01,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
4,633807,2016-02-18,FD2298,Yes,Yes,MS,44.90,female,24,2011-02-01,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000659,0.0,0.0


### Check for Matches and Unique Values in Various Metadata

Matches in Species

In [711]:
# standardize the feature names first across datasets
# Levi
standardized_data_2 = standardize_species(levi_species_unique_df.columns.tolist())
lowercase_standardized_data_2_features = {result.lower() for result in standardized_data_2}

# Zhou
lowercase_data_features = {col.lower().strip() for col in zhou_species_df.columns} 

# look for matches
matches = sorted(lowercase_standardized_data_2_features.intersection(lowercase_data_features))
print(matches)
print(f"Number of matches: {len(matches)}")

['acidaminococcus intestini', 'actinomyces neuii', 'actinomyces radingae', 'actinomyces turicensis', 'adlercreutzia equolifaciens', 'akkermansia muciniphila', 'alistipes finegoldii', 'alistipes indistinctus', 'alistipes inops', 'alistipes putredinis', 'alistipes shahii', 'alistipes timonensis', 'allisonella histaminiformans', 'anaerococcus tetradius', 'anaerostipes hadrus', 'anaerotruncus colihominis', 'atopobium minutum', 'atopobium vaginae', 'bacteroides cellulosilyticus', 'bacteroides coprocola', 'bacteroides coprophilus', 'bacteroides finegoldii', 'bacteroides fragilis', 'bacteroides ovatus', 'bacteroides plebeius', 'bacteroides salyersiae', 'bacteroides stercoris', 'bacteroides thetaiotaomicron', 'bacteroides uniformis', 'bacteroides vulgatus', 'bifidobacterium adolescentis', 'bifidobacterium angulatum', 'bifidobacterium animalis', 'bifidobacterium bifidum', 'bifidobacterium longum', 'blautia hydrogenotrophica', 'blautia obeum', 'butyricimonas synergistica', 'butyrivibrio crossotu

Matches in Genus

In [716]:
# look for matches
matches = sorted(levi_genus_unique_df.columns.intersection(zhou_genus_df.columns))
print(matches)
print(f"Number of matches: {len(matches)}")

['acetobacter', 'acidaminococcus', 'actinomyces', 'adlercreutzia', 'akkermansia', 'alistipes', 'allisonella', 'anaerococcus', 'anaeromassilibacillus', 'anaerosphaera', 'anaerostipes', 'anaerotruncus', 'atopobium', 'azospirillum', 'bacteroides', 'bifidobacterium', 'bilophila', 'blautia', 'butyricicoccus', 'butyricimonas', 'butyrivibrio', 'campylobacter', 'catenibacterium', 'clostridium', 'collinsella', 'coprobacillus', 'coprobacter', 'coprococcus', 'coraliomargarita', 'corynebacterium', 'cryptobacterium', 'desulfovibrio', 'dialister', 'dorea', 'eggerthella', 'eisenbergiella', 'enterobacter', 'enterococcus', 'erysipelatoclostridium', 'escherichia', 'eubacterium', 'faecalibacterium', 'finegoldia', 'flavonifractor', 'fusicatenibacter', 'fusobacterium', 'gardnerella', 'gemella', 'haemophilus', 'holdemanella', 'hungatella', 'intestinibacter', 'intestinimonas', 'klebsiella', 'lachnoclostridium', 'lachnospira', 'lactobacillus', 'lactococcus', 'lawsonella', 'megamonas', 'megasphaera', 'methanob

In [713]:
# look at various other information related to metadata counts
print(merged_samples_species['treatment_status'].value_counts().reindex(['Treated', 'Untreated', 'Control']))
print(merged_samples_species['site'].value_counts().reindex(['San Francisco', 'Boston', 'New York', 'Edinburgh', 'Buenos Aires', 'San Sebastian', 'Pittsburgh']))
print(merged_samples_species['sex'].value_counts().reindex(['M', 'F']))
print(pd.crosstab(merged_samples_species['treatment_status'], merged_samples_species['sex']))

treatment_status
Treated      367
Untreated    209
Control      576
Name: count, dtype: int64
site
San Francisco    328
Boston            84
New York         118
Edinburgh        262
Buenos Aires     258
San Sebastian     78
Pittsburgh        24
Name: count, dtype: int64
sex
M    551
F    601
Name: count, dtype: int64
sex                 F    M
treatment_status          
Control           201  375
Treated           255  112
Untreated         145   64
